In [2]:
import requests

url = "https://clinicaltrials.gov/api/v2/studies"
params = {
    "query.cond": "cancer",
    "pageSize": 100,
    "format": "json"
}

r = requests.get(url, params=params)
trials = r.json()["studies"]

print(f"Got {len(trials)} trials")
print(trials[0]["protocolSection"]["eligibilityModule"]["eligibilityCriteria"][:500])

Got 100 trials
Key Inclusion Criteria:

* Subject signed a valid, Institutional Review Board (IRB)/Ethics Committee (EC)-approved informed consent form.
* Male or female age 18 or older when written informed consent is obtained.
* Study candidate is willing and able to comply with all protocol-required procedures and assessments.
* Study candidate either

  1. is scheduled for or plans to be scheduled for a biopsy of the liver OR
  2. has a previous histologically or cytologically confirmed diagnosis of adenoc


In [3]:
import json

with open("data/trials_raw.json", "w") as f:
    json.dump(trials, f, indent=2)

print("Data saved!")
print(f"Total trials saved: {len(trials)}")

Data saved!
Total trials saved: 100


In [ ]:
import pandas as pd

records = []

for trial in trials:
    try:
        nct_id = trial["protocolSection"]["identificationModule"]["nctId"]
        title = trial["protocolSection"]["identificationModule"]["briefTitle"]
        eligibility = trial["protocolSection"]["eligibilityModule"]["eligibilityCriteria"]
        records.append({
            "nct_id": nct_id,
            "title": title,
            "eligibility_text": eligibility
        })
    except KeyError:
        continue

df = pd.DataFrame(records)
print(f"Extracted {len(df)} trials")
print(df.head(3))

In [4]:
import pandas as pd

records = []

for trial in trials:
    try:
        nct_id = trial["protocolSection"]["identificationModule"]["nctId"]
        title = trial["protocolSection"]["identificationModule"]["briefTitle"]
        eligibility = trial["protocolSection"]["eligibilityModule"]["eligibilityCriteria"]
        records.append({
            "nct_id": nct_id,
            "title": title,
            "eligibility_text": eligibility
        })
    except KeyError:
        continue

df = pd.DataFrame(records)
print(f"Extracted {len(df)} trials")
print(df.head(3))

Extracted 100 trials
        nct_id                                              title  \
0  NCT05189171  MicroOrganoSphere (MOS) Drug Screen Pilot Tria...   
1  NCT03320044  Early Diagnosis of Small Pulmonary Nodules by ...   
2  NCT01205711  Irinotecan Hydrochloride, Fluorouracil, and Le...   

                                    eligibility_text  
0  Key Inclusion Criteria:\n\n* Subject signed a ...  
1  Inclusion Criteria:\n\n1. Sign informed consen...  
2  DISEASE CHARACTERISTICS:\n\n* Histologically o...  


In [5]:
df.to_csv("data/trials_clean.csv", index=False)
print("Clean data saved to data/trials_clean.csv!")

Clean data saved to data/trials_clean.csv!


In [6]:
# How long is the eligibility text typically?
df["text_length"] = df["eligibility_text"].str.len()

print("=== Data Summary ===")
print(f"Total trials: {len(df)}")
print(f"Average text length: {df['text_length'].mean():.0f} characters")
print(f"Shortest text: {df['text_length'].min()} characters")
print(f"Longest text: {df['text_length'].max()} characters")
print("\n=== Sample eligibility text ===")
print(df['eligibility_text'][0][:800])

=== Data Summary ===
Total trials: 100
Average text length: 2513 characters
Shortest text: 81 characters
Longest text: 12964 characters

=== Sample eligibility text ===
Key Inclusion Criteria:

* Subject signed a valid, Institutional Review Board (IRB)/Ethics Committee (EC)-approved informed consent form.
* Male or female age 18 or older when written informed consent is obtained.
* Study candidate is willing and able to comply with all protocol-required procedures and assessments.
* Study candidate either

  1. is scheduled for or plans to be scheduled for a biopsy of the liver OR
  2. has a previous histologically or cytologically confirmed diagnosis of adenocarcinoma of the colon and/or rectum that is metastatic to the liver.

Key Exclusion Criteria:

* If already scheduled for a biopsy of the liver: liver biopsy was ordered to help diagnose, determine the severity of, or treat a disease that is unrelated to colorectal cancer (e.g., nonalcoholic fatty l


In [7]:
import re

labeled_criteria = []

for _, row in df.iterrows():
    text = row["eligibility_text"]
    nct_id = row["nct_id"]
    
    # Split into inclusion and exclusion sections
    # Try to find where exclusion starts
    split_patterns = [
        r"(?i)exclusion criteria",
        r"(?i)key exclusion",
        r"(?i)exclude"
    ]
    
    exclusion_start = len(text)  # default: no exclusion found
    for pattern in split_patterns:
        match = re.search(pattern, text)
        if match:
            exclusion_start = match.start()
            break
    
    inclusion_text = text[:exclusion_start]
    exclusion_text = text[exclusion_start:]
    
    # Split into individual bullet points
    def extract_bullets(block):
        lines = re.split(r'\n[\*\-\d•]+\.?\s+', block)
        return [l.strip() for l in lines if len(l.strip()) > 30]
    
    for criterion in extract_bullets(inclusion_text):
        labeled_criteria.append({
            "nct_id": nct_id,
            "criterion": criterion,
            "label": "inclusion"
        })
    
    for criterion in extract_bullets(exclusion_text):
        labeled_criteria.append({
            "nct_id": nct_id,
            "criterion": criterion,
            "label": "exclusion"
        })

labeled_df = pd.DataFrame(labeled_criteria)
print(f"Total labeled criteria: {len(labeled_df)}")
print(f"\nLabel distribution:")
print(labeled_df["label"].value_counts())
print(f"\nSample inclusion:")
print(labeled_df[labeled_df["label"]=="inclusion"]["criterion"].iloc[0])
print(f"\nSample exclusion:")
print(labeled_df[labeled_df["label"]=="exclusion"]["criterion"].iloc[0])

Total labeled criteria: 1309

Label distribution:
label
exclusion    687
inclusion    622
Name: count, dtype: int64

Sample inclusion:
Subject signed a valid, Institutional Review Board (IRB)/Ethics Committee (EC)-approved informed consent form.

Sample exclusion:
If already scheduled for a biopsy of the liver: liver biopsy was ordered to help diagnose, determine the severity of, or treat a disease that is unrelated to colorectal cancer (e.g., nonalcoholic fatty liver disease, chronic hepatitis B or C, autoimmune hepatitis, alcoholic liver disease, primary biliary cirrhosis, primary sclerosing cholangitis, hemochromatosis, Wilson's disease, etc.).


In [8]:
# Save labeled data
labeled_df.to_csv("data/labeled_criteria.csv", index=False)
print("Labeled data saved!")
print(f"\nFinal dataset shape: {labeled_df.shape}")

Labeled data saved!

Final dataset shape: (1309, 3)
